**A conditional chain in LangChain uses if-else logic to route execution dynamically to different sub-chains or runnables based on input values or intermediate results**.

* **How It Works**
    Modern LangChain implements conditional branching primarily through the RunnableBranch class from langchain_core.runnables.C
    
    - **ondition-Runnable Pairs**: It accepts a list of tuples containing a condition (a boolean function or expression) and a corresponding runnable chain.
    - **Default Fallback**: It requires a default fallback runnable that executes if none of the explicit conditions evaluate to true.
    - **Evaluation Order**: It evaluates conditions sequentially and executes the first branch where the condition returns True

In [1]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv

## This function will load all the variable from .env file and will make them available in
## os.environ directory (env_variablea)
load_dotenv()

import os 

if os.environ.get("OPENAI_API_KEY"):
    print("✅ OPENAI_API_KEY Exists.")
else:
    raise ValueError("❌ OPENAI_API_KEY Not Found...")


✅ OPENAI_API_KEY Exists.


In [2]:
llm_openai = ChatOpenAI(model='gpt-5-mini',
                        temperature=0)

In [3]:
from pydantic import BaseModel
from typing import Literal
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

class MovieReview(BaseModel):
    movie_summary: Literal["positive", "negative"]

llm_structured_output = llm_openai.with_structured_output(MovieReview)

### **Chain with Conditional Chains**

In [4]:
# Task 1: Prompt
prompt_template = ChatPromptTemplate.from_messages([
    ('system', 'You are a movie Review evaluator'),
    ('human', "please categorise the movie as positive or negative: {input}")
])

In [5]:
# Task 2: LLM
llm_structured_output = llm_openai.with_structured_output(MovieReview)

In [6]:
# Task 3; Output parser
from langchain_core.output_parsers import StrOutputParser

str_parser = StrOutputParser()

In [7]:
# task 4: Custom Runnable
from langchain_core.runnables import RunnableLambda

def pydantic_json(text: MovieReview) -> str:
    return text.model_dump_json()

pydantic_json_lambda = RunnableLambda(pydantic_json)

### Conditional Chain 1

In [8]:
# Task 1: Prompt
linkedin_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a Linkedin Post Generator'),
    ('human', 'crearte a post for the following topic: {text}')
])

# Task 2: LLM
llm_openai = ChatOpenAI(model='gpt-5-mini',
                        temperature=0)

# Task 3; Output parser
str_parser = StrOutputParser()

# Linkedin Chain
chain = linkedin_prompt | llm_openai | str_parser

### Conditional Chain 2

In [9]:
def insta_chain(text:dict):
    
    text = text['text']
    
    # Task 1: Prompt
    insta_prompt = ChatPromptTemplate.from_messages([
        ('system', 'You are a Instagram Post Generator'),
        ('human', 'crearte a post for the following topic: {text}')
    ])

    # Task 2: LLM
    llm_openai = ChatOpenAI(model='gpt-5-mini',
                            temperature=0)

    # Task 3; Output parser
    str_parser = StrOutputParser()
    
    chain_insta = insta_prompt | llm_openai | str_parser
    result = chain_insta.invoke(text)
    
    return result

insta_chain_runnable = RunnableLambda(insta_chain)

### Final Orchestration

In [10]:
# Conditional Chains
from langchain_core.runnables import RunnableBranch

conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain),
    (lambda x: "negative" in x, insta_chain_runnable), # can be eliminated here
    insta_chain_runnable   ## giving default runnable
)

final_orchestrator = prompt_template | llm_structured_output | pydantic_json_lambda | conditional_chain

In [14]:
final_orchestrator.invoke({'input':"I loved it"})

'Short version (quick LinkedIn share)\n----------------------------------\nJust finished a movie that left me inspired — not because of special effects, but because of the way it celebrated resilience, teamwork, and quiet courage.\n\nIn a few lines: a determined protagonist faces setback after setback, learns from unexpected allies, and turns small daily choices into lasting change. Uplifting, human, and full of practical lessons for how we work together.\n\nTakeaway: persistence + empathy > perfection. What recent film reminded you why people matter at work? #Leadership #Resilience #Learning\n\nLong version (reflective LinkedIn post)\n--------------------------------------\nI watched a film last night that wasn’t just entertaining — it was a timely reminder of what makes great teams and great leaders.\n\nSynopsis (short): the story follows a main character who navigates major obstacles, finds strength in unlikely collaborators, and ultimately achieves progress through steady effort, c